# Gold Layer: 도로 점수화

열 위험도 · 노면 거칠기 · 쿠션감 3가지 지표로 반려견 산책 경로를 점수화합니다.

**실행 환경**: Databricks (Spark + DBFS)

## 1. 열 위험도 (`heat_risk`)

surface 태그가 있으면 재질 기반으로 직접 산출하고, 없으면 토성 + 배수등급 조합으로 추정합니다.  
둘 다 없는 경우엔 highway 유형으로 fallback합니다 (residential은 대부분 아스팔트로 간주).

In [0]:
from pyspark.sql.functions import col, when, lit, coalesce, desc

gold_df = spark.table("silver.walk_features")

# 토성별 열 보유력 점수 (Max 30) — 미매칭은 None 처리 후 coalesce로 기본값 적용
# 식질 30 / 미사식양질·역질·사력질 25 / 사질·식양질 20 / 미사사양질 15 / 사양질 10
gold_df = gold_df.withColumn("heat_soil_val",
    when(col("soil_type") == "식질", 30)
    .when(col("soil_type").isin("미사식양질", "역질", "사력질"), 25)
    .when(col("soil_type").isin("사질", "식양질"), 20)
    .when(col("soil_type") == "미사사양질", 15)
    .when(col("soil_type") == "사양질", 10)
    .otherwise(None)
)

# 배수등급별 증발 냉각 점수 (Max 25) — 매우양호(건조·뜨거움) 25 → 불량(수분 냉각) 8
gold_df = gold_df.withColumn("heat_drain_val",
    when(col("drainage_class") == "매우양호", 25)
    .when(col("drainage_class") == "양호", 20)
    .when(col("drainage_class").isin("약간양호", "약간 양호"), 15)
    .when(col("drainage_class").isin("약간불량", "약간 불량"), 12)
    .when(col("drainage_class") == "매우 불량", 10)
    .when(col("drainage_class") == "불량", 8)
    .otherwise(None)
)

gold_df = gold_df.withColumn("heat_risk",
    # 초고위험: 인공 포장재
    when(col("surface").isin(
        "asphalt", "concrete", "paved", "concrete:plates",
        "Braille_blocks", "paving_stones"
    ), 70 + coalesce(col("heat_drain_val"), lit(5)))

    # 고위험: 돌·석재
    .when(col("surface").isin(
        "cobblestone", "stone", "cobblestone:flattened", "sett", "gravel", "rock"
    ), 45 + coalesce(col("heat_drain_val"), lit(5)))

    # 중위험: 인공 탄성·다져진 길
    .when(col("surface").isin(
        "urethane", "tartan", "rubber", "compacted"
    ), 30 + coalesce(col("heat_drain_val"), lit(5)))

    # surface 없음 — 토성·배수 조합 또는 highway 추정
    .otherwise(
        when(col("heat_soil_val").isNotNull() | col("heat_drain_val").isNotNull(),
             coalesce(col("heat_soil_val"), lit(15)) + coalesce(col("heat_drain_val"), lit(10)) + lit(15))
        .otherwise(
            when(col("highway") == "residential", 60)
            .when(col("highway") == "footway", 55)
            .when(col("highway") == "path", 50)
            .when(col("highway") == "track", 48)
            .otherwise(52)
        )
    )
)

print("✅ Heat Risk 산출 완료")
print("heat_risk 점수 분포")
gold_df.groupBy("heat_risk").count().orderBy(desc("heat_risk")).show(truncate=False)


✅ Heat Risk 산출 완료
heat_risk 점수 분포
+---------+-----+
|heat_risk|count|
+---------+-----+
|95       |492  |
|90       |2494 |
|85       |360  |
|82       |841  |
|78       |6    |
|75       |8058 |
|70       |6    |
|65       |3436 |
|60       |35706|
|57       |44   |
|55       |42474|
|52       |2845 |
|50       |22152|
|48       |1202 |
|47       |4372 |
|45       |12241|
|43       |1    |
|42       |525  |
|40       |1594 |
|38       |4    |
+---------+-----+
only showing top 20 rows



## 2. 노면 거칠기 (`roughness_score`)

OSM smoothness 실측값을 최우선으로 쓰고, 없으면 자갈함량 → 토성 → surface → highway 순으로 추론합니다.  
경사가 급할수록 동일 노면도 관절 부하가 커지므로 slope 구간별 가중치를 곱합니다.

In [0]:
from pyspark.sql.functions import col, when, lit, coalesce, trim, round, desc

# OSM smoothness 실측값 — 있으면 최우선 적용
# excellent 15 / good 30 / intermediate 50 / bad 80 / very_bad 100
gold_df = gold_df.withColumn("measured_roughness",
    when(col("smoothness") == "excellent",    15)
    .when(col("smoothness") == "good",        30)
    .when(col("smoothness") == "intermediate",50)
    .when(col("smoothness") == "bad",         80)
    .when(col("smoothness") == "very_bad",   100)
    .otherwise(None)
)

# smoothness 없을 때 자갈함량 → 토성 → surface → highway 순으로 단계적 추론
gold_df = gold_df.withColumn("inferred_roughness",
    # Level 1. 자갈 함량 — 발바닥 상처·미끄러짐 위험 직접 반영
    when(trim(col("gravel_content")) == ">35",  80)
    .when(trim(col("gravel_content")) == "10-35",60)
    .when(trim(col("gravel_content")) == "<10",  30)

    # Level 2. 토성 — 점토질: 건조 시 균열·우천 시 미끄러움 / 모래질: 발목 빠짐
    .when(col("soil_type").isin("식질", "식양질", "미사식양질"), 50)
    .when(col("soil_type") == "사질",                           40)
    .when(col("soil_type").isin("사양질", "미사사양질"),         20)

    # Level 3. surface 재질 기반
    .when(col("surface") == "cobblestone:flattened",                          55)  # 평탄화 자갈
    .when(col("surface").isin("rock", "stone", "cobblestone", "gravel", "unpaved"), 80)
    .when(col("surface") == "Braille_blocks",                                 65)  # 점자블록 요철
    .when(col("surface").isin("earth", "ground", "sand", "compacted"),        50)
    .when(col("surface") == "concrete:plates",                                35)  # 판형, 이음새 있음
    .when(col("surface").isin("paving_stones", "paved"),                      30)
    .when(col("surface").isin("asphalt", "concrete"),                         20)
    .when(col("surface").isin("urethane", "wood"),                            15)

    # Level 4. 데이터 완전 부재 — highway 타입으로 기본값 할당
    .otherwise(
        when(col("highway") == "steps",                            70)
        .when(col("highway") == "residential",                     40)
        .when(col("highway").isin("footway", "path", "track"),    55)
        .otherwise(45)
    )
)

# 실측값 우선, 없으면 추론값 / 경사도 가중치 적용
gold_df = gold_df.withColumn("base_roughness",
    coalesce(col("measured_roughness"), col("inferred_roughness"))
).withColumn("roughness_score",
    when(col("avg_slope") >= 15, col("base_roughness") * 1.3)
    .when(col("avg_slope") >   8, col("base_roughness") * 1.2)
    .when(col("avg_slope") >   3, col("base_roughness") * 1.1)
    .otherwise(col("base_roughness"))
)

# 100점 상한 및 소수점 1자리 정리
gold_df = gold_df.withColumn("roughness_score",
    when(col("roughness_score") > 100, 100)
    .otherwise(round(col("roughness_score"), 1))
)

print("✅ Roughness 산출 완료")
gold_df.groupBy("roughness_score").count().orderBy(desc("roughness_score")).show(truncate=False)


✅ Roughness 산출 완료
+---------------+-----+
|roughness_score|count|
+---------------+-----+
|100.0          |358  |
|96.0           |193  |
|88.0           |944  |
|80.0           |762  |
|78.0           |8801 |
|72.0           |5245 |
|70.0           |1006 |
|66.0           |620  |
|60.5           |16306|
|60.0           |300  |
|55.0           |34474|
|50.0           |267  |
|49.5           |988  |
|45.0           |374  |
|40.0           |32484|
|39.0           |145  |
|36.0           |1304 |
|35.0           |15   |
|33.0           |17171|
|30.0           |16316|
+---------------+-----+
only showing top 20 rows



## 3. 쿠션감 (`cushion_score`)

인공 탄성 포장재(urethane 등)는 최고점 100으로 고정하고, 자연 비포장로는 토심 베이스에 토성 보정값을 더합니다.  
토심이 깊을수록 충격 흡수에 유리하고, 모래질일수록 완충 효과가 커집니다.

In [0]:
from pyspark.sql.functions import col, when, trim, lit

# 토심 베이스 점수 — 깊을수록 충격 흡수력 높음
# 100cm 초과 90 / 50-100cm 70 / 20-50cm 40 / 기타 25
cushion_base = (
    when(trim(col("soil_depth")) == "100초과", 90)
    .when(trim(col("soil_depth")) == "50-100",  70)
    .when(trim(col("soil_depth")) == "20-50",   40)
    .otherwise(25)
)

# 토성 보정값 — 입자 클수록(모래질) 완충↑, 작을수록(점토질) 딱딱함
cushion_soil = (
    when(trim(col("soil_type")).isin("미사사양질", "사양질"),     15)
    .when(trim(col("soil_type")) == "사질",                       10)
    .when(trim(col("soil_type")).isin("식양질", "미사식양질"),    -5)
    .when(trim(col("soil_type")) == "식질",                      -15)
    .otherwise(0)
)

gold_df = gold_df.withColumn("cushion_score",
    when(col("surface").isin("urethane", "tartan", "rubber", "wood"), 100)

    # 자연 비포장 — 토심+토성 조합으로 정밀 계산
    .when(col("surface").isin("unpaved", "ground", "dirt", "earth", "sand", "compacted", "grass"),
          (cushion_base + cushion_soil).cast("int"))

    .when(col("surface") == "cobblestone:flattened", 20)
    .when(col("surface") == "Braille_blocks",        15)  # 요철 구조, 충격 흡수 낮음
    .when(col("surface").isin("cobblestone", "stone"),12)
    .when(col("surface").isin(
        "asphalt", "concrete", "concrete:plates",
        "paved", "paving_stones", "sett"),             10)

    # surface 없음 — highway 타입으로 추정
    .otherwise(
        when(col("highway") == "steps",                   5)
        .when(col("highway") == "residential",            15)
        .when(col("highway").isin("footway", "path"),    45)
        .when(col("highway") == "track",                  55)
        .otherwise(25)
    )
)

# 0~100 클리핑
gold_df = gold_df.withColumn("cushion_score",
    when(col("cushion_score") > 100, 100)
    .when(col("cushion_score") <   0,  0)
    .otherwise(col("cushion_score"))
)

print("✅ Cushion Score 산출 완료")
gold_df.groupBy("cushion_score").count().orderBy(col("cushion_score").desc()).show()


✅ Cushion Score 산출 완료
+-------------+-----+
|cushion_score|count|
+-------------+-----+
|          100|  473|
|           85| 1002|
|           80|   26|
|           70|    6|
|           65|   62|
|           55| 1556|
|           45|70909|
|           40|  182|
|           35|   40|
|           25| 5781|
|           20|   26|
|           15|49320|
|           12|   81|
|           10|12258|
|            5| 1573|
+-------------+-----+



In [0]:
# surface 추정 — 실측값 우선, 없으면 cushion·roughness 점수로 지면 성격 추론
gold_df = gold_df.withColumn(
    "inferred_surface",
    when(col("surface").isNotNull(), col("surface"))
    .when((col("cushion_score") >= 70), "푹신한 토양")
    .when((col("roughness_score") >= 60), "거친 노면")
    .otherwise("일반 노면")
).withColumn(
    # 위험 요소 우선 판정 후 긍정 등급 순으로 분류
    "final_safety_grade",
    when(col("roughness_score") >= 80,                           "위험 (발바닥 부상 주의)")
    .when(col("heat_risk") >= 80,                                "주의 (여름철 화상 주의)")
    .when(col("inferred_surface").isin("urethane", "tartan", "rubber"), "매우 안전 (탄성 포장)")
    .when(col("cushion_score") >= 75,                            "매우 안전 (푹신한 흙길)")
    .when(col("inferred_surface").isin("asphalt", "concrete", "paving_stones", "sett"), "평범 (아스팔트/보도블록)")
    .otherwise("보통 (일반 노면)")
)


In [0]:
from pyspark.sql.functions import array, array_compact, concat_ws, concat, col, coalesce, round

# UI 필터링용 속성 배열 — None 항목은 array_compact로 제거
gold_df = gold_df.withColumn(
    "filter_attributes",
    array_compact(array(
        col("slope_type"),
        when(col("highway") == "steps", lit("계단있음")).otherwise(lit("계단없음")),
        when(col("cushion_score")   >= 70, lit("푹신함")),
        when(col("heat_risk")       >= 70, lit("지면뜨거움")),
        when(col("roughness_score") >= 60, lit("거친길"))
    ))
)

# cushion + slope 조합 보행감 메시지
# highway 추정 구간(cushion 15 고정)의 반복 문구 방지를 위해 slope_type도 함께 분기
gold_df = gold_df.withColumn(
    "cushion_msg",
    when(col("cushion_score") >= 85,
         "매우 뛰어난 쿠션감을 제공하여 노령견에게 최적입니다.")
    .when(col("cushion_score") > 55,
         "보행감이 좋아 강아지가 편안하게 산책할 수 있는 구간입니다.")
    .when((col("cushion_score") <= 15) & (col("slope_type") == "평지"),
         "지면은 단단한 편이나 평탄하여 가벼운 산책에 적합합니다.")
    .when((col("cushion_score") <= 15) & (col("slope_type") == "완만"),
         "지면이 다소 딱딱하므로 완만한 경사를 따라 천천히 걷는 것을 권장합니다.")
    .when((col("cushion_score") <= 15) & (col("slope_type") == "급경사"),
         "지면이 딱딱하고 경사가 가팔라 관절에 무리가 갈 수 있으니 짧은 산책 위주를 추천합니다.")
    .otherwise("지면이 다소 딱딱하여 장시간 산책 시 주의가 필요합니다.")
)

# RAG용 서술형 설명 — LLM 검색 기반 경로 추천에 활용
gold_df = gold_df.withColumn(
    "road_description",
    trim(concat_ws(" ",
        lit("[경로 정보]"), lit("이 산책로는"),
        coalesce(col("final_safety_grade"), lit("일반적인")), lit("상태의 구간입니다."),

        lit("\n[경사도 및 계단 여부]"),
        when(col("highway") == "steps",
             lit("이 구간은 계단으로 구성되어 있어 보행 시 주의가 필요합니다."))
        .otherwise(
            concat(lit("평균 경사도가 약 "), coalesce(col("avg_slope"), lit(0.0)),
                   lit("%인 "), coalesce(col("slope_type"), lit("평탄한")), lit(" 구간입니다."))
        ),

        lit("\n[지면 상태]"),
        when(col("inferred_surface").isNotNull(),
             concat(lit("실제 재질은 "), col("inferred_surface"), lit("이며, ")))
        .otherwise(lit("지형 분석 결과")),
        coalesce(col("soil_type"), lit("복합")), lit("토양 성분을 포함하고 있습니다."),

        lit("\n[보행감 정보]"), col("cushion_msg"),

        lit("\n[태그]"), concat_ws(", ", col("filter_attributes"))
    ))
).drop("cushion_msg")

spark.sql("CREATE DATABASE IF NOT EXISTS gold")

gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.scored")

print("✅ gold.scored 저장 완료")
display(gold_df.select(
    "highway", 
    "surface", 
    "heat_risk", 
    "cushion_score", 
    "filter_attributes", 
    "road_description"
).limit(10))

✅ gold.scored 저장 완료


highway,surface,heat_risk,cushion_score,filter_attributes,road_description
footway,null,55,45,"List(평지, 계단없음)","[경로 정보] 이 산책로는 보통 (일반 노면) 상태의 구간입니다. [경사도 및 계단 여부] 평균 경사도가 약 2.5%인 평지 구간입니다. [지면 상태] 실제 재질은 일반 노면이며, Unknown 토양 성분을 포함하고 있습니다. [보행감 정보] 지면이 다소 딱딱하여 장시간 산책 시 주의가 필요합니다. [태그] 평지, 계단없음"
footway,null,55,45,"List(평지, 계단없음)","[경로 정보] 이 산책로는 보통 (일반 노면) 상태의 구간입니다. [경사도 및 계단 여부] 평균 경사도가 약 2.5%인 평지 구간입니다. [지면 상태] 실제 재질은 일반 노면이며, Unknown 토양 성분을 포함하고 있습니다. [보행감 정보] 지면이 다소 딱딱하여 장시간 산책 시 주의가 필요합니다. [태그] 평지, 계단없음"
residential,null,60,15,"List(평지, 계단없음)","[경로 정보] 이 산책로는 보통 (일반 노면) 상태의 구간입니다. [경사도 및 계단 여부] 평균 경사도가 약 1.5%인 평지 구간입니다. [지면 상태] 실제 재질은 일반 노면이며, Unknown 토양 성분을 포함하고 있습니다. [보행감 정보] 지면은 단단한 편이나 평탄하여 가벼운 산책에 적합합니다. [태그] 평지, 계단없음"
footway,asphalt,75,10,"List(평지, 계단없음, 지면뜨거움)","[경로 정보] 이 산책로는 평범 (아스팔트/보도블록) 상태의 구간입니다. [경사도 및 계단 여부] 평균 경사도가 약 2.5%인 평지 구간입니다. [지면 상태] 실제 재질은 asphalt이며, Unknown 토양 성분을 포함하고 있습니다. [보행감 정보] 지면은 단단한 편이나 평탄하여 가벼운 산책에 적합합니다. [태그] 평지, 계단없음, 지면뜨거움"
residential,null,60,15,"List(평지, 계단없음)","[경로 정보] 이 산책로는 보통 (일반 노면) 상태의 구간입니다. [경사도 및 계단 여부] 평균 경사도가 약 1.5%인 평지 구간입니다. [지면 상태] 실제 재질은 일반 노면이며, Unknown 토양 성분을 포함하고 있습니다. [보행감 정보] 지면은 단단한 편이나 평탄하여 가벼운 산책에 적합합니다. [태그] 평지, 계단없음"
path,null,45,45,"List(급경사, 계단없음, 거친길)","[경로 정보] 이 산책로는 보통 (일반 노면) 상태의 구간입니다. [경사도 및 계단 여부] 평균 경사도가 약 45.0%인 급경사 구간입니다. [지면 상태] 실제 재질은 거친 노면이며, 사양질 토양 성분을 포함하고 있습니다. [보행감 정보] 지면이 다소 딱딱하여 장시간 산책 시 주의가 필요합니다. [태그] 급경사, 계단없음, 거친길"
footway,null,50,45,"List(급경사, 계단없음, 거친길)","[경로 정보] 이 산책로는 보통 (일반 노면) 상태의 구간입니다. [경사도 및 계단 여부] 평균 경사도가 약 45.0%인 급경사 구간입니다. [지면 상태] 실제 재질은 거친 노면이며, 사양질 토양 성분을 포함하고 있습니다. [보행감 정보] 지면이 다소 딱딱하여 장시간 산책 시 주의가 필요합니다. [태그] 급경사, 계단없음, 거친길"
path,null,50,45,"List(완만, 계단없음, 거친길)","[경로 정보] 이 산책로는 보통 (일반 노면) 상태의 구간입니다. [경사도 및 계단 여부] 평균 경사도가 약 3.5%인 완만 구간입니다. [지면 상태] 실제 재질은 거친 노면이며, Unknown 토양 성분을 포함하고 있습니다. [보행감 정보] 지면이 다소 딱딱하여 장시간 산책 시 주의가 필요합니다. [태그] 완만, 계단없음, 거친길"
path,null,50,45,"List(완만, 계단없음, 거친길)","[경로 정보] 이 산책로는 보통 (일반 노면) 상태의 구간입니다. [경사도 및 계단 여부] 평균 경사도가 약 3.5%인 완만 구간입니다. [지면 상태] 실제 재질은 거친 노면이며, Unknown 토양 성분을 포함하고 있습니다. [보행감 정보] 지면이 다소 딱딱하여 장시간 산책 시 주의가 필요합니다. [태그] 완만, 계단없음, 거친길"
path,earth,50,25,"List(완만, 계단없음)","[경로 정보] 이 산책로는 보통 (일반 노면) 상태의 구간입니다. [경사도 및 계단 여부] 평균 경사도가 약 3.5%인 완만 구간입니다. [지면 상태] 실제 재질은 earth이며, Unknown 토양 성분을 포함하고 있습니다. [보행감 정보] 지면이 다소 딱딱하여 장시간 산책 시 주의가 필요합니다. [태그] 완만, 계단없음"
